In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Measure the tokens, time and cost of an agent turn

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### What a turn costs

An agent turn is several model calls. The coordinator decides what to do, tools run, and the model is called again with their results. Each call is billed by its tokens:

- **Input tokens**: everything the model reads on that call: the instructions, the conversation, the tool results.
- **Output tokens**: the text and tool calls it writes. Thinking tokens, the model's reasoning before it answers, are billed as output.
- **Cached tokens**: input that repeats between calls, such as the instructions, billed at a lower rate when [context caching](https://cloud.google.com/vertex-ai/generative-ai/docs/context-cache/context-cache-overview) applies.

Every model response carries these counts in its `usage_metadata`. The data side of the bill is BigQuery, which charges for the bytes each query scans.

### Thinking levels

Gemini models can think before they answer. The [thinking level](https://cloud.google.com/vertex-ai/generative-ai/docs/thinking) sets how much: a higher level can give better reasoning on hard problems, and costs more thinking tokens and more time. The store agent runs at medium, except the writer of the opening plan, which runs at low.

### Objectives

In this tutorial, you will learn where the tokens and the time in an agent turn go, and what changes them.

You will complete the following tasks:

- Record the tokens and seconds of every model call in a turn, including calls inside tools
- Put a price on a turn and project a month of use
- Compare thinking levels on one prompt, and on the plan writer
- Read the BigQuery side of the bill from the job history

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This notebook runs against the store data you loaded in the earlier notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The agent's code lives one folder up from this notebook
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, and the Gen AI SDK logs a note whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

import time

import pandas as pd
import yaml
from google import genai
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.runners import InMemoryRunner
from google.cloud import bigquery
from google.genai import types

from agents.cymbal_store_ops.agent import create_app
from agents.cymbal_store_ops.config import load_env_config

## Record every model call

An ADK [plugin](https://google.github.io/adk-docs/plugins/) runs callbacks around every model call in the app, including the calls made by agents that run inside tools, such as the opening plan's writer. This plugin notes the start time before each call and, after it, records the agent's name, the seconds taken and the token counts from `usage_metadata`:

In [4]:
class UsageRecorder(BasePlugin):
    """Records the agent, seconds and token counts of every model call."""

    def __init__(self):
        super().__init__(name="usage_recorder")
        self.calls = []
        self._started = {}

    async def before_model_callback(self, *, callback_context, llm_request):
        key = (callback_context.invocation_id, callback_context.agent_name)
        self._started[key] = time.perf_counter()

    async def after_model_callback(self, *, callback_context, llm_response):
        usage = llm_response.usage_metadata
        if usage is None or llm_response.partial:
            return
        key = (callback_context.invocation_id, callback_context.agent_name)
        self.calls.append(
            {
                "agent": callback_context.agent_name,
                "seconds": round(time.perf_counter() - self._started.pop(key), 1),
                "input": usage.prompt_token_count or 0,
                "cached": usage.cached_content_token_count or 0,
                "output": usage.candidates_token_count or 0,
                "thinking": usage.thoughts_token_count or 0,
            }
        )

Build the agent, add the plugin and define a helper that runs one question in a new session as Dana, the store manager, and returns the calls it recorded:

In [5]:
def build_runner() -> tuple[InMemoryRunner, UsageRecorder]:
    """Build the store agent with a fresh usage recorder."""
    app = create_app(log_events=False)
    recorder = UsageRecorder()
    app.plugins.append(recorder)
    return InMemoryRunner(app=app), recorder


async def measure(question: str) -> pd.DataFrame:
    """Run one question as Dana and return one row per model call."""
    runner, recorder = build_runner()
    session = await runner.session_service.create_session(
        app_name=runner.app_name,
        user_id="dana",
        state={
            "user:user_id": "U-M014",
            "user:store_id": "S-014",
            "user:role": "store_manager",
            "user:first_name": "Dana",
        },
    )
    message = types.Content(role="user", parts=[types.Part(text=question)])
    started = time.perf_counter()
    async for _ in runner.run_async(user_id="dana", session_id=session.id, new_message=message):
        pass
    print(f"{question}\n{time.perf_counter() - started:.1f} seconds in total")
    return pd.DataFrame(recorder.calls)

### Measure two turns

Measure an ordinary stock question and the opening plan:

In [6]:
stock_turn = await measure("How many units of Lumière Hydra Cream do we have, and where are they?")
stock_turn

How many units of Lumière Hydra Cream do we have, and where are they?
25.4 seconds in total


,agent,seconds,input,cached,output,thinking
0,store_manager_agent,9.3,7430,3066,24,264
1,store_manager_agent,2.4,7842,6163,25,136
2,store_manager_agent,8.3,9080,6141,282,973


In [7]:
opening_turn = await measure("Morning. Just opened up. What should I be on top of first?")
opening_turn

Morning. Just opened up. What should I be on top of first?
11.6 seconds in total


,agent,seconds,input,cached,output,thinking
0,store_manager_agent,3.2,7428,0,29,161
1,plan_writer,5.1,12209,0,678,0


Input tokens are most of the total on every call: each time the model is called, it reads the instructions, the conversation and every tool result so far. In this run the stock question took about 25 seconds: three calls by `store_manager_agent`, each reading 7,400 to 9,100 input tokens, with tool calls between them. The `cached` column shows input read from the context cache; on the second and third calls about 6,100 of those tokens were cached.

The opening plan took about 12 seconds. Its long call is `plan_writer`, which read 12,209 tokens of evidence and wrote a 678-token plan with no thinking tokens, because it runs at low. The tables vary a little from run to run.

## Put a price on a turn

Fill in the prices for the model from the [Vertex AI pricing page](https://cloud.google.com/vertex-ai/generative-ai/pricing), in US dollars per one million tokens. Prices change, so they are inputs here rather than numbers in the code.

In [8]:
PRICE_INPUT = 0.0  # @param {type: "number"}
PRICE_CACHED_INPUT = 0.0  # @param {type: "number"}
PRICE_OUTPUT = 0.0  # @param {type: "number"}


def turn_cost(calls: pd.DataFrame) -> float:
    """The cost of one turn in US dollars. Thinking tokens are billed as output."""
    uncached = (calls["input"] - calls["cached"]).sum()
    return (
        uncached * PRICE_INPUT
        + calls["cached"].sum() * PRICE_CACHED_INPUT
        + (calls["output"] + calls["thinking"]).sum() * PRICE_OUTPUT
    ) / 1_000_000


summary = pd.DataFrame(
    [
        {
            "turn": name,
            "model calls": len(calls),
            "input": calls["input"].sum(),
            "output + thinking": (calls["output"] + calls["thinking"]).sum(),
            "usd": round(turn_cost(calls), 5),
        }
        for name, calls in [("stock question", stock_turn), ("opening plan", opening_turn)]
    ]
)
if PRICE_INPUT == 0:
    print("Fill in the prices above to see the usd column.")
summary

Fill in the prices above to see the usd column.


,turn,model calls,input,output + thinking,usd
0,stock question,3,24352,1704,0.0
1,opening plan,2,19637,868,0.0


With the prices left at 0, the `usd` column is 0. The token totals are real: in this run the stock question read 24,352 input tokens and wrote 1,704 output and thinking tokens over three calls, and the opening plan read 19,637 and wrote 868 over two. Multiply by the prices you entered to check the `usd` column. The counts vary a little from run to run.

### Project a month

Multiply by how often people use the agent. This is a planning number, not a forecast: the opening plan happens once a day for each manager, while ordinary questions happen many times.

In [9]:
STORES = 1400  # @param {type: "integer"}
OPENINGS_PER_STORE_PER_DAY = 1  # @param {type: "integer"}
QUESTIONS_PER_STORE_PER_DAY = 20  # @param {type: "integer"}

per_day = STORES * (
    OPENINGS_PER_STORE_PER_DAY * turn_cost(opening_turn)
    + QUESTIONS_PER_STORE_PER_DAY * turn_cost(stock_turn)
)
if PRICE_INPUT == 0:
    print("Fill in the prices in the previous step to see the projection.")
else:
    print(f"Per day: ${per_day:,.2f}   Per 30 days: ${per_day * 30:,.2f}")

Fill in the prices in the previous step to see the projection.


## Compare thinking levels

### On one prompt

Send the same question straight to Gemini at three thinking levels and compare the thinking tokens and the time. At `LOW` the model may answer without a separate thinking step, so it reports no thinking tokens. When many people share a project, a request can be refused with `429 RESOURCE_EXHAUSTED`; the client's retry options wait and try again. The three calls take about 30 seconds:

In [10]:
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="global",
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=4)),
)
model = load_env_config().model
prompt = (
    "A store has 9 pickup orders (13 units) due between 9:30 and 11:00 AM. Picking takes 6 minutes an order. "
    "Priya starts at 9:00 and has a break from 10:30 to 10:45. Can she finish every order on time? Answer in two sentences."
)

rows = []
for level in (types.ThinkingLevel.LOW, types.ThinkingLevel.MEDIUM, types.ThinkingLevel.HIGH):
    started = time.perf_counter()
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_level=level)),
    )
    rows.append(
        {
            "level": level.name,
            "seconds": round(time.perf_counter() - started, 1),
            "thinking tokens": response.usage_metadata.thoughts_token_count,
            "answer": response.text,
        }
    )
pd.DataFrame(rows)

,level,seconds,thinking tokens,answer
0,LOW,7.8,NaN,"No, Priya cannot finish all the orders on time..."
1,MEDIUM,5.1,813.0,"Yes, Priya can easily finish every order on ti..."
2,HIGH,20.1,2980.0,"Yes, Priya can finish every order on time beca..."


Look at the thinking tokens and the seconds. In this run `LOW` reported no thinking tokens (`NaN`), `MEDIUM` 813 and `HIGH` 2,980, and `HIGH` took about 20 seconds. The time of one call varies, so here `MEDIUM` was faster than `LOW`.

Read the answers too. In this run `LOW` answered no, while `MEDIUM` and `HIGH` answered yes: 9 orders at 6 minutes is 54 minutes of picking, which fits before her break. The wording and sometimes the verdict change from run to run.

### On the plan writer

The environment file sets the level for the whole agent and, separately, for the plan writer:

In [11]:
env_config = yaml.safe_load((REPO_ROOT / "agents/cymbal_store_ops/config/envs/dev.yaml").read_text())
{key: env_config[key] for key in ("model", "thinking_level", "plan_writer_thinking_level")}

{'model': 'gemini-3.8-flash',
 'thinking_level': 'medium',
 'plan_writer_thinking_level': 'low'}

The writer was moved to low after a benchmark: five openings at each level, with every plan checked against the evidence it was given. At low, the writer took about 6 seconds instead of about 24, and every plan was grounded.

Measure it yourself. The opening plan is a workflow, `daily_briefing`: three readers gather the evidence and the writer turns it into the plan. Run the workflow on its own, once with the writer at low and once at medium. The `PLAN_WRITER_THINKING_LEVEL` variable overrides the environment file, and `load_env_config` caches the configuration it reads, so clear the cache before you build the workflow again. The two runs take about 40 seconds:

In [12]:
from agents.cymbal_store_ops.sub_agents.daily_briefing import make_daily_briefing


async def time_writer(level: str) -> pd.DataFrame:
    """Run the opening-plan workflow with the writer at one thinking level."""
    os.environ["PLAN_WRITER_THINKING_LEVEL"] = level
    load_env_config.cache_clear()
    recorder = UsageRecorder()
    runner = InMemoryRunner(
        agent=make_daily_briefing(model=load_env_config().model),
        app_name="daily_briefing",
        plugins=[recorder],
    )
    session = await runner.session_service.create_session(
        app_name="daily_briefing",
        user_id="dana",
        state={"user:user_id": "U-M014", "user:store_id": "S-014", "user:role": "store_manager"},
    )
    message = types.Content(role="user", parts=[types.Part(text="Opening priorities for the store")])
    async for _ in runner.run_async(user_id="dana", session_id=session.id, new_message=message):
        pass
    return pd.DataFrame(recorder.calls).assign(writer_level=level)


writer_levels = pd.concat([await time_writer("low"), await time_writer("medium")], ignore_index=True)

os.environ.pop("PLAN_WRITER_THINKING_LEVEL")
load_env_config.cache_clear()
writer_levels

,agent,seconds,input,cached,output,thinking,writer_level
0,plan_writer,5.7,12201,0,578,0,low
1,plan_writer,26.9,12201,0,612,3334,medium


In this run the writer at low took 5.7 seconds with no thinking tokens, and at medium 26.9 seconds with 3,334 thinking tokens. Both read the same 12,201 input tokens and wrote about 600 output tokens, so the difference is the thinking. Your times will differ.

One run of each is not a benchmark: the time of a single call varies. The repository's [`eval/benchmark_writer.py`](../eval/benchmark_writer.py) runs several openings at each level and checks every plan against its evidence.

## Read the BigQuery side of the bill

Every query the agent runs carries labels: the namespace, the environment and the name of the tool. That makes the BigQuery job history a record of cost by tool. `JOBS_BY_USER` lists the queries you ran, which includes the agent turns in this notebook. The deployed agent's queries run as its service account, through the MCP server; `JOBS_BY_PROJECT` shows those if you have the BigQuery Resource Viewer role. BigQuery on-demand pricing charges for bytes scanned, so group the jobs from the last day by tool:

In [13]:
bq = bigquery.Client(project=PROJECT_ID)

sql = f"""
SELECT
  IFNULL((SELECT value FROM UNNEST(labels) WHERE key = 'tool'), '(data load and setup)') AS tool,
  COUNT(*) AS jobs,
  ROUND(SUM(total_bytes_billed) / 1e6, 2) AS mb_billed,
  ROUND(AVG(TIMESTAMP_DIFF(end_time, start_time, MILLISECOND))) AS avg_ms
FROM `{PROJECT_ID}.region-us`.INFORMATION_SCHEMA.JOBS_BY_USER
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND job_type = 'QUERY'
  AND EXISTS (SELECT 1 FROM UNNEST(labels) WHERE key = 'ns' AND value = @namespace)
GROUP BY tool
ORDER BY mb_billed DESC
"""
job = bq.query(
    sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("namespace", "STRING", WORKSHOP_NAMESPACE)]
    ),
)
pd.DataFrame([dict(row) for row in job.result()])

,tool,jobs,mb_billed,avg_ms
0,query_store_data,94,1237.32,459.0
1,check_store_stock,138,597.69,208.0
2,(data load and setup),40,440.40,1735.0
3,get_operations_context,475,377.49,169.0
4,get_bopis_demand,242,377.49,200.0
5,get_replenishment_status,216,251.66,180.0
6,get_shrink_signals,59,251.66,174.0
7,get_task_status,181,230.69,177.0
8,get_task_history,61,230.69,174.0
9,get_osa_exceptions,63,188.74,203.0


Each row is one tool, over every query in your namespace in the last 24 hours, so the counts include the earlier notebooks and anything else you ran. In this run `query_store_data`, the open query tool, billed the most (about 1,240 MB over 94 jobs) and `get_operations_context` ran most often (475 jobs). `healthcheck` bills nothing. Your numbers depend on what you ran today.

Each query is also capped: the agent's configuration sets `maximum_bytes_billed` on every job, so one question cannot scan the whole dataset.

## What changes the cost

| Lever | Where | What it changes |
|---|---|---|
| Thinking level | `thinking_level` and `plan_writer_thinking_level` in `config/envs/dev.yaml` | Thinking tokens and time on each call |
| Context caching | `ContextCacheConfig` in `agents/cymbal_store_ops/agent.py` | Repeated input billed at the cached rate |
| What the model reads | Finished specialist conversations are pruned; the plan writer gets compact tables | Input tokens |
| Reports instead of rows | Large results go to a report in the app; the model gets a summary | Input tokens |
| Query size | `maximum_bytes_billed` and `max_query_result_rows` in the environment file | BigQuery bytes |

## Cleaning up

This notebook creates no cloud resources. The sessions lived in memory and end when you restart the kernel.

## What's next

- [Thinking in Gemini](https://cloud.google.com/vertex-ai/generative-ai/docs/thinking)
- [Context caching](https://cloud.google.com/vertex-ai/generative-ai/docs/context-cache/context-cache-overview)
- [Cost and quotas](../docs/COST_AND_QUOTAS.md) for this repository
- Back to the [workshop notebooks](README.md)